# 07 Climate Projections for Water
**Series:** Pine Ridge Hydrology                                                                              
**Author:** Lilly Jones, PhD                                                                                                        
**Primary Focus:** Pine Ridge Reservation/Oglala Sioux Tribe                                                              
**Collective:** Oglala Lakota                                                                                
**Data Source:** MACAv2-METDATA via OPeNDAP                                                                            

## From Historical to Future
Notebooks 01–06 characterized the water system as it has behaved since 2000.
This notebook asks: what do climate projections say about how that system
will change?

For water managers on Pine Ridge, the most consequential
projected changes are:

1. **Higher temperatures** : more evapotranspiration demand, accelerated
   aquifer depletion, more frequent extreme heat that stresses livestock
   and reduces water quality
2. **Precipitation intensity vs. reliability** : possibly more intense
   events but more dry days between them; episodic recharge rather than
   slow steady infiltration that replenishes aquifers
3. **Reduced snowpack** : earlier snowmelt means less late-spring baseflow
   and more early summer dry periods
4. **Longer droughts** : both higher temperatures and reduced precipitation
   contribute to more frequent and persistent PDSI drought conditions

## Scenario Framing
We present two RCP scenarios:
- **RCP 4.5** : moderate emissions reduction (still significant warming)
- **RCP 8.5** : business-as-usual high emissions

The difference between scenarios is a policy choice.
Both show warming; the magnitude depends on what happens to emissions.

## Learning Objectives

By the end of this notebook, learners will be able to:

- distinguish a climate scenario, model projection, and forecast
- compare time periods and scenarios without implying certainty
- explain why a single-model result is insufficient for a research-grade local projection

## Prerequisites and Timing

Allow approximately 75–100 minutes. Before beginning, activate the repository environment, read the series governance statement, and complete the preceding notebook where applicable. Work in pairs and rotate analyst, data-steward, skeptic, and documentarian roles.

## Governance Checkpoint

This notebook uses public environmental data describing Oglala Lakota lands and waters. Public availability does not establish permission for every reuse or interpretation. Do not add OST-controlled data, sensitive locations, or community knowledge. Results are educational and screening-level pending OLC/OST review.

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml
from scipy import stats
import warnings

from src.constants import (
    OUTPUTS_DIR, FIGURES_DIR,
    REPO_ROOT as _REPO_ROOT, CACHE_DIR,
)
from src.config import load_config, streamflow_site_ids, streamflow_site_names

CONFIG = load_config()
STUDY_BBOX = tuple(CONFIG["study_area"]["hydrologic_context_bbox"])
STUDY_NAMES = [CONFIG["study_area"]["people"]]
STUDY_CENTROIDS = {CONFIG["study_area"]["people"]: CONFIG["study_area"]["centroid"]}
PINE_RIDGE_STREAMGAGES = {
    site["name"]: str(site["id"]) for site in CONFIG["usgs_streamflow_sites"]
}

from src.indicators import theilsen_trend
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
%matplotlib inline

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

with open(_REPO_ROOT / "config" / "config.yaml") as f:
    CONFIG = yaml.safe_load(f)

# Primary site centroids
SITES = {"Oglala Lakota": STUDY_CENTROIDS["Oglala Lakota"]}

# MACAv2 parameters
MACA_BASE  = "http://thredds.northwestknowledge.net:8080/thredds/dodsC/agg_macav2metdata_"
HIST_START = 1950
HIST_END   = 2005   
PROJ_START = 2006
PROJ_END   = 2099
SCENARIOS  = ["rcp45", "rcp85"]
# single model now for speed; ensemble average is preferred in research
MODEL      = "BNU-ESM"   

print(f"Primary sites: {list(SITES.keys())}")
print(f"MACAv2 model: {MODEL} | Scenarios: {SCENARIOS}")

In [ ]:
# Print data sovereignty acknowledgement at the top of every notebook
print_data_acknowledgment(source_keys=["maca_climate"])

## Fetch MACAv2 Temperature and Precipitation

In [ ]:
def fetch_maca_point(
    lat: float, lon: float,
    variable: str,
    scenario: str,
    model: str = MODEL,
    site_name: str = "site",
) -> pd.DataFrame:
    cache_key  = f"maca_{site_name}_{variable}_{scenario}.csv"
    cache_file = CACHE_DIR / cache_key

    if cache_file.exists():
        return pd.read_csv(cache_file, parse_dates=["date"])

    if scenario == "historical":
        url = (f"http://thredds.northwestknowledge.net:8080/thredds/dodsC/"
               f"agg_macav2metdata_{variable}_{model}_r1i1p1_historical"
               f"_1950_2005_CONUS_monthly.nc")
    else:
        url = (f"http://thredds.northwestknowledge.net:8080/thredds/dodsC/"
               f"agg_macav2metdata_{variable}_{model}_r1i1p1_{scenario}"
               f"_2006_2099_CONUS_monthly.nc")

    try:
        ds     = xr.open_dataset(url, engine="netcdf4",
                                 decode_times=True, use_cftime=True)
        lon360 = lon % 360
        ds_pt  = ds.sel(lon=lon360, lat=lat, method="nearest")
        var_name = [v for v in ds_pt.data_vars][0]

        # Convert cftime (NoLeap calendar) to pandas datetime
        # by manually building year-month from the cftime index
        da       = ds_pt[var_name]
        # array of cftime objects
        times    = da.time.values   
        dates    = pd.to_datetime([
            f"{t.year}-{t.month:02d}-01" for t in times
        ])
        values   = da.values

        series = pd.DataFrame({"date": dates, variable: values})
        series = series.dropna(subset=[variable])
        series.to_csv(cache_file, index=False)
        ds.close()
        return series

    except Exception as e:
        warnings.warn(f"MACAv2 {variable} {scenario}: {e}", UserWarning)
        return pd.DataFrame()

In [ ]:
# Fetch for Pine Ridge (primary site)
pr_lat = SITES["Oglala Lakota"]["lat"]
pr_lon = SITES["Oglala Lakota"]["lon"]

maca_data = {}

for scenario in SCENARIOS:
    print(f"\nFetching {scenario}...")
    tmax = fetch_maca_point(pr_lat, pr_lon, "tasmax", scenario,
                             site_name="pine_ridge")
    pr   = fetch_maca_point(pr_lat, pr_lon, "pr",    scenario,
                             site_name="pine_ridge")
    if not tmax.empty:
        tmax["tmax_f"] = (tmax["tasmax"] - 273.15) * 9/5 + 32
    maca_data[scenario] = {"tmax": tmax, "pr": pr}
    print(f"  Tmax: {len(tmax):,} months")
    print(f"  Precip: {len(pr):,} months")

In [ ]:
# Diagnose what came back
for scenario, data in maca_data.items():
    print(f"\n{scenario}:")
    for var, df in data.items():
        print(f"  {var}: {len(df)} rows")
        if not df.empty:
            print(f"    columns: {df.columns.tolist()}")
            print(f"    head: {df.head(2).to_string()}")
        else:
            print(f"    EMPTY: fetch failed silently")

# Test one URL directly to see the actual error
# re-enable warnings temporarily
warnings.filterwarnings("default")  

url = f"http://thredds.northwestknowledge.net:8080/thredds/dodsC/agg_macav2metdata_tasmax_{MODEL}_rcp45_2006_CONUS_daily.nc"
print(f"\nTesting URL:\n{url}")
try:
    ds = xr.open_dataset(url, engine="netcdf4")
    print(f"Dataset opened: {list(ds.data_vars)}")
    print(f"Coords: {list(ds.coords)}")
    ds.close()
except Exception as e:
    print(f"Failed: {e}")

## Water-Relevant Metrics

In [ ]:
# Compute annual metrics relevant to water management
annual_results = {}

for scenario, data in maca_data.items():
    tmax_df = data["tmax"]
    pr_df   = data["pr"]

    records = []
    years   = sorted(tmax_df["date"].dt.year.unique()) if not tmax_df.empty else []

    for year in years:
        rec = {"year": year, "scenario": scenario}

        if not tmax_df.empty:
            yr_tmax = tmax_df[tmax_df["date"].dt.year == year]
            rec["mean_tmax_f"]       = yr_tmax["tmax_f"].mean()
            # Heat stress days: >100°F threshold relevant for livestock and humans
            rec["heat_stress_days"]   = (yr_tmax["tmax_f"] > 100).sum()
            # Extreme heat: >110°F
            rec["extreme_heat_days"]  = (yr_tmax["tmax_f"] > 110).sum()

        if not pr_df.empty:
            yr_pr = pr_df[pr_df["date"].dt.year == year]
            rec["annual_precip_mm"]   = yr_pr["pr"].sum()
            # Dry days: < 1mm/day
            rec["dry_days"]           = (yr_pr["pr"] < 1).sum()
            # Heavy precip days: > 25mm (potential for erosion, flash flood)
            rec["heavy_precip_days"]  = (yr_pr["pr"] > 25).sum()

        records.append(rec)

    if records:
        annual_results[scenario] = pd.DataFrame(records)

for scenario, df in annual_results.items():
    print(f"\n{scenario}: {len(df)} annual records")
    print(df.describe().round(2).to_string())

## Visualizations

In [ ]:
# Temperature and heat stress trend
if annual_results:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    SCENARIO_COLORS = {"rcp45": "#E67E22", "rcp85": "#C0392B"}
    SCENARIO_LABELS = {"rcp45": "RCP 4.5 (moderate)", "rcp85": "RCP 8.5 (high)"}

    metrics = [
        ("mean_tmax_f",      "Mean annual tmax (°F)",        axes[0, 0]),
        ("heat_stress_days", "Heat stress days (> 100°F)",   axes[0, 1]),
        ("annual_precip_mm", "Annual precipitation (mm)",    axes[1, 0]),
        ("dry_days",         "Annual dry days (< 1 mm/day)", axes[1, 1]),
    ]

    for col, label, ax in metrics:
        for scenario, df in annual_results.items():
            if col not in df.columns:
                continue
            color = SCENARIO_COLORS[scenario]

            # 10-year rolling mean
            rolling = df[col].rolling(10, center=True, min_periods=5).mean()
            ax.plot(df["year"], df[col], color=color, alpha=0.2, linewidth=0.8)
            ax.plot(df["year"], rolling, color=color, linewidth=2.5,
                    label=SCENARIO_LABELS[scenario])

        ax.set_ylabel(label, fontsize=9)
        ax.set_xlabel("Year", fontsize=8)
        ax.legend(fontsize=7)
        despine(ax)

    plt.suptitle(
        "MACAv2 Climate Projections for Pine Ridge, Oglala Lakota Nation\n"
        f"Model: {MODEL} | Thin line = annual | Thick = 10-year rolling mean",
        fontsize=10, fontweight="bold",
    )
    plt.tight_layout()
    try:
        fig.savefig(FIGURES_DIR/"07_climate_projections.png",
                    dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()

In [ ]:
# Scenario divergence chart: the policy choice made visible
if len(annual_results) == 2 and "mean_tmax_f" in list(annual_results.values())[0].columns:
    df45 = annual_results["rcp45"].set_index("year")["mean_tmax_f"]
    df85 = annual_results["rcp85"].set_index("year")["mean_tmax_f"]
    common = df45.index.intersection(df85.index)

    r45  = df45.loc[common].rolling(10, center=True, min_periods=5).mean()
    r85  = df85.loc[common].rolling(10, center=True, min_periods=5).mean()

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.fill_between(common, r45.values, r85.values,
                    alpha=0.3, color="#C0392B",
                    label="Range between scenarios")
    ax.plot(common, r45.values, color="#E67E22", linewidth=2.5,
            label="RCP 4.5 (moderate emissions)")
    ax.plot(common, r85.values, color="#C0392B", linewidth=2.5,
            label="RCP 8.5 (high emissions)")

    # End-of-century comparison
    late_45 = r45.loc[2070:2099].mean()
    late_85 = r85.loc[2070:2099].mean()
    ax.annotate(
        f"2070–2099\nRCP 4.5: {late_45:.1f}°F",
        xy=(2095, late_45), fontsize=8, color="#E67E22",
        xytext=(-80, 10), textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="#E67E22"),
    )
    ax.annotate(
        f"RCP 8.5: {late_85:.1f}°F",
        xy=(2095, late_85), fontsize=8, color="#C0392B",
        xytext=(-80, -20), textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="#C0392B"),
    )

    ax.set_xlabel("Year", fontsize=10)
    ax.set_ylabel("Annual mean maximum temperature (°F)", fontsize=10)
    ax.set_title(
        "Temperature Scenario Divergence for Pine Ridge, Oglala Lakota Nation\n"
        "The gap between lines represents the effect of emissions reductions",
        fontsize=10, fontweight="bold",
    )
    ax.legend(fontsize=9)
    despine(ax)
    plt.tight_layout()
    try:
        fig.savefig(FIGURES_DIR/"07_scenario_divergence.png",
                    dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()

    print(f"Scenario divergence by end of century:")
    print(f"  RCP 4.5 mean tmax (2070–2099): {late_45:.1f}°F")
    print(f"  RCP 8.5 mean tmax (2070–2099): {late_85:.1f}°F")
    print(f"  Difference: {late_85 - late_45:.1f}°F")
    print(f"  This gap is the result of emissions choices made in coming decades.")

## Exports

In [ ]:
for scenario, df in annual_results.items():
    out = OUTPUTS_DIR/f"maca_{scenario}_annual.csv"
    df.to_csv(out, index=False)
    print(f"Exported to {out.name}")

In [ ]:
print(generate_citations(["maca_climate"]))

## Learner Checkpoint

Rewrite one projected change as a bounded statement naming the model, scenario, period, spatial representation, and major uncertainty.

## Interpretation Protocol

Before writing a conclusion, separate:

1. **Observation:** what the computed public data show, including unit, period, spatial scope, and missingness.
2. **Interpretation:** a plausible explanation, stated with uncertainty.
3. **Additional evidence:** literature, local monitoring, expertise, or validation needed to evaluate that explanation.
4. **Decision authority:** who is authorized to approve publication, thresholds, or management action.

Do not convert monitoring absence, association, a screening flag, or scenario output into a causal, regulatory, health, policy, or community conclusion.

## Contribution Activity

Improve one uncertainty statement, scenario label, time-period comparison, or recommendation for multi-model analysis. Review the change with a partner and record what became clearer or more defensible.

## Evidence Record and Next Step

Record one regenerated result, its source and scope, one transformation, one limitation, and one question requiring more evidence or local knowledge.

Synthesize the series by tracing one claim back through its artifact, configuration, public source, assumptions, and required review authority.